# GCN Cloud Notebook

This notebook is configured for Google Colab using GitHub as the code source. It clones the repo into `/content`, keeps Google Drive optional, and runs the cleaned lattice stiffness workflow from `colab_gnn_stiffness_prototype.py`.

In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules
print(f'Running in Colab: {IN_COLAB}')

if IN_COLAB:
    %pip -q install torch-geometric
else:
    print('Colab dependency install cell skipped.')

In [ ]:
REPO_URL = 'https://github.com/aadams2006/NSF-REU-Summer-26.git'
REPO_DIR = '/content/NSF-REU-Summer-26'

if IN_COLAB:
    import os
    if not os.path.isdir(REPO_DIR):
        !git clone {REPO_URL} {REPO_DIR}
    else:
        print(f'Repo already exists at {REPO_DIR}')
else:
    print('Git clone cell skipped outside Colab.')

In [ ]:
from datetime import datetime
from getpass import getpass
from pathlib import Path
import os
import sys

USE_DRIVE_FOR_DATA = False
SAVE_OUTPUTS_TO_DRIVE = True
PUSH_RESULTS_TO_GITHUB = False
PUSH_MODEL_TO_GITHUB = False
DRIVE_DATA_ROOT = '/content/drive/MyDrive/lattice_data'
DRIVE_OUTPUT_ROOT = '/content/drive/MyDrive/gnn_prototype_outputs/baseline'
GIT_RESULTS_SUBDIR = 'active_projects/voronoi_lattice_pipeline/gnn_prototype/outputs/baseline'
GIT_BRANCH = 'main'
GITHUB_TOKEN = os.environ.get('GITHUB_TOKEN', '').strip()
GIT_COMMIT_USERNAME = os.environ.get('GIT_COMMIT_USERNAME', '').strip()
GIT_COMMIT_EMAIL = os.environ.get('GIT_COMMIT_EMAIL', '').strip()
RUN_STAMP = datetime.now().strftime('%Y%m%d_%H%M%S')

if IN_COLAB:
    repo_root = Path(REPO_DIR).resolve()
else:
    repo_root = Path.cwd().resolve()

pipeline_root = repo_root / 'active_projects' / 'voronoi_lattice_pipeline'
module_dir = pipeline_root / 'gnn_prototype'
if not (module_dir / 'colab_gnn_stiffness_prototype.py').is_file():
    raise FileNotFoundError(f'Module not found at {module_dir}')

if str(module_dir) not in sys.path:
    sys.path.insert(0, str(module_dir))

if IN_COLAB and (USE_DRIVE_FOR_DATA or SAVE_OUTPUTS_TO_DRIVE):
    from google.colab import drive
    drive.mount('/content/drive')

if USE_DRIVE_FOR_DATA:
    if not IN_COLAB:
        raise RuntimeError('USE_DRIVE_FOR_DATA is only supported in Colab.')
    drive_data_root = Path(DRIVE_DATA_ROOT)
    train_root = drive_data_root / 'Randomness_Sweep'
    predict_root = drive_data_root / 'Lattice_Guess_Prediction_Input_Data'
else:
    train_root = pipeline_root / 'source_archives' / 'lattice_data' / 'Randomness_Sweep'
    predict_root = pipeline_root / 'datasets' / 'Lattice_Guess_Prediction_Input_Data'

if IN_COLAB and SAVE_OUTPUTS_TO_DRIVE:
    output_root = Path(DRIVE_OUTPUT_ROOT)
else:
    output_root = Path('/content/gnn_outputs/baseline') if IN_COLAB else pipeline_root / 'gnn_prototype' / 'outputs' / 'baseline'

git_output_root = repo_root / GIT_RESULTS_SUBDIR
output_dir = output_root / f'run_{RUN_STAMP}'
output_dir.mkdir(parents=True, exist_ok=True)
(output_root / 'latest_run.txt').write_text(str(output_dir), encoding='utf-8')

if PUSH_RESULTS_TO_GITHUB:
    if not GITHUB_TOKEN:
        GITHUB_TOKEN = getpass('Enter GitHub token: ').strip()
    if not GIT_COMMIT_USERNAME:
        GIT_COMMIT_USERNAME = input('Enter Git commit username or display name: ').strip()
    if not GIT_COMMIT_EMAIL:
        GIT_COMMIT_EMAIL = input('Enter Git commit email (GitHub noreply or verified email): ').strip()

print(f'Repo root: {repo_root}')
print(f'Pipeline root: {pipeline_root}')
print(f'Train data: {train_root}')
print(f'Prediction data: {predict_root}')
print(f'Output root: {output_root}')
print(f'Current run dir: {output_dir}')
print(f'Git output root: {git_output_root}')
print(f'Push results to GitHub: {PUSH_RESULTS_TO_GITHUB}')
print(f'GitHub token loaded: {bool(GITHUB_TOKEN)}')
print(f'Git commit username loaded: {bool(GIT_COMMIT_USERNAME)}')
print(f'Git commit email loaded: {bool(GIT_COMMIT_EMAIL)}')

In [ ]:
import pandas as pd
import shutil
import subprocess
from IPython.display import display

from colab_gnn_stiffness_prototype import (
    TrainingConfig,
    SimpleGNN,
    create_data_loaders,
    default_data_roots,
    evaluate_model,
    load_lattice_dataset,
    normalize_feature_splits,
    plot_prediction_splits,
    plot_training_history,
    predict_on_directory,
    save_run_artifacts,
    set_seed,
    split_dataset,
    summarize_metrics,
    train_model,
)

In [ ]:
config = TrainingConfig()
set_seed(config.seed)

print(f'Device: {config.device}')
print(f'Batch size: {config.batch_size}')
print(f'Hidden dim: {config.hidden_dim}')
print(f'Epochs: {config.total_epochs}')

In [ ]:
dataset = load_lattice_dataset(train_root)
train_data, val_data, test_data = split_dataset(dataset, seed=config.seed)
scaler = normalize_feature_splits(train_data, val_data, test_data)
train_loader, val_loader, test_loader = create_data_loaders(
    train_data,
    val_data,
    test_data,
    batch_size=config.batch_size,
)

model = SimpleGNN(
    input_dim=train_data[0].x.shape[1],
    hidden_dim=config.hidden_dim,
)

print(f'Train samples: {len(train_data)}')
print(f'Validation samples: {len(val_data)}')
print(f'Test samples: {len(test_data)}')
model

In [ ]:
history = train_model(model, train_loader, val_loader, config)
plot_training_history(history, config.epochs_phase1)

In [ ]:
metrics_by_split = {}
split_results = []

for split_name, loader in (("Train", train_loader), ("Validation", val_loader), ("Test", test_loader)):
    predictions, ground_truth, metrics = evaluate_model(model, loader, device=config.device)
    metrics_by_split[split_name] = metrics
    split_results.append((split_name, predictions, ground_truth))

metrics_frame = summarize_metrics(metrics_by_split)
metrics_frame

In [ ]:
plot_prediction_splits(split_results)

In [ ]:
prediction_results, prediction_metrics = predict_on_directory(
    model,
    predict_root,
    scaler,
    device=config.device,
)

prediction_summary = pd.Series(prediction_metrics, name='Prediction Set')
display(prediction_results.head())
display(prediction_summary)

saved_dir = save_run_artifacts(
    output_dir,
    model,
    scaler,
    history,
    metrics_by_split,
    prediction_results=prediction_results,
)

saved_files = sorted(path.name for path in saved_dir.iterdir() if path.is_file())
print(f'Saved artifacts to {saved_dir}')
print(f'Latest run pointer: {output_root / "latest_run.txt"}')
print('Saved files:')
for name in saved_files:
    print(f' - {name}')

if PUSH_RESULTS_TO_GITHUB:
    if not IN_COLAB:
        raise RuntimeError('GitHub auto-push is only intended for the Colab clone workflow.')
    if not GITHUB_TOKEN:
        raise ValueError('Set GITHUB_TOKEN before enabling PUSH_RESULTS_TO_GITHUB.')

    git_run_dir = git_output_root / saved_dir.name
    git_run_dir.mkdir(parents=True, exist_ok=True)

    files_to_push = ['metrics_summary.csv', 'prediction_results.csv']
    if PUSH_MODEL_TO_GITHUB:
        files_to_push.append('lattice_gnn_model.pt')

    for file_name in files_to_push:
        source_path = saved_dir / file_name
        if source_path.is_file():
            shutil.copy2(source_path, git_run_dir / file_name)

    (git_output_root / 'latest_run.txt').write_text(str(git_run_dir.relative_to(repo_root)), encoding='utf-8')

    subprocess.run(['git', '-C', str(repo_root), 'config', 'user.name', GIT_COMMIT_USERNAME], check=True)
    subprocess.run(['git', '-C', str(repo_root), 'config', 'user.email', GIT_COMMIT_EMAIL], check=True)

    remote_url = subprocess.run(
        ['git', '-C', str(repo_root), 'remote', 'get-url', 'origin'],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
    auth_url = remote_url.replace('https://', f'https://{GITHUB_TOKEN}@', 1)
    subprocess.run(['git', '-C', str(repo_root), 'remote', 'set-url', 'origin', auth_url], check=True)

    try:
        subprocess.run(['git', '-C', str(repo_root), 'add', str(git_run_dir), str(git_output_root / 'latest_run.txt')], check=True)
        diff_result = subprocess.run(
            ['git', '-C', str(repo_root), 'diff', '--cached', '--quiet'],
            check=False,
        )
        if diff_result.returncode == 0:
            print('No GitHub changes to commit.')
        else:
            commit_message = f'Add GCN cloud results for {saved_dir.name}'
            subprocess.run(['git', '-C', str(repo_root), 'commit', '-m', commit_message], check=True)
            subprocess.run(['git', '-C', str(repo_root), 'push', 'origin', GIT_BRANCH], check=True)
            print(f'Pushed results to GitHub under {git_run_dir.relative_to(repo_root)}')
    finally:
        subprocess.run(['git', '-C', str(repo_root), 'remote', 'set-url', 'origin', remote_url], check=True)

In [ ]:
if IN_COLAB and not SAVE_OUTPUTS_TO_DRIVE:
    from google.colab import files
    archive_path = '/content/gnn_outputs_baseline.zip'
    !cd /content && zip -qr gnn_outputs_baseline.zip gnn_outputs/baseline
    files.download(archive_path)
else:
    print(f'Outputs are in {output_dir}')